# Phylogeny in Sound — a FiftyOne demo (audio-toolkit build)

**Watch the tree of life emerge from audio embeddings — and hear the clips right in the App.**

This notebook builds an interactive [FiftyOne](https://docs.voxel51.com) app on top of
[`rinvictor/bioacoustic-phylogeny-embeddings`](https://github.com/rinvictor/bioacoustic-phylogeny-embeddings) —
the code/data for the preprint *"Phylogenetic signal in marine mammal and bird
vocalizations captured by audio foundation models"* (Rincón Yepes, 2026, arXiv:2607.22458).

The paper's finding, in one sentence: **general-purpose audio models (CLAP, AST)
encode the evolutionary tree about as well as bioacoustic specialists (BEATs-bio,
BirdNET) do**, even though nobody trained them to. This notebook lets you *see* that
claim — and *hear* the vocalizations — instead of reading a correlation coefficient.

### How audio works in this build

Audio playback and spectrograms happen **inside the FiftyOne App**, via one installed
plugin: **[`@roboav8r/fiftyone-audio-toolkit`](https://github.com/roboav8r/fiftyone-audio-toolkit)**.
It bundles exactly what this demo needs:

1. a **spectrogram sample renderer** — audio files report media type `unknown`, so the
   renderer takes over their grid/modal tiles and draws a log-power (magma) spectrogram
   *in-browser* (a self-contained STFT — no PNG pre-render, no server);
2. an **`<audio>` player + synced playhead** in the modal — double-click a tile, press
   play, watch the playhead sweep across the spectrogram; and
3. an **Audio Embeddings Search panel** + CLAP operators — query by a selected sample, an
   uploaded clip, or free text, then explore the similarity region.

The key consequence: **each sample's media is the `.wav` file itself**, not a
pre-rendered image. The toolkit draws the spectrogram and provides the player; the
notebook just builds the dataset and the embedding plots.

> **Open-source compatible.** The toolkit's manifest declares a FiftyOne *Enterprise*
> version (`fiftyone: version: ">=2.18.0"`). Enterprise and open-source use parallel
> version numbers for the same codebase, so that literal string won't validate against an
> open-source install — but the features it needs (the `SampleRenderer` component API) are
> present in current open-source FiftyOne. We relax the manifest's version pin to the
> open-source line during install (one `sed` line, below), and everything works: grid
> spectrograms, modal player with playhead, and the search panel — all working on
> current open-source FiftyOne.

### Target environment (verified against Voxel51's install docs)
- **FiftyOne:** latest open-source release (the **1.20.x** line; there is no open-source
  2.x — those numbers are FiftyOne Enterprise).
- **Python:** **3.12**. FiftyOne supports 3.10–3.13, but the audio stack is the tighter
  constraint: `librosa`'s `numba`/`llvmlite` only ship arm64 wheels through 3.12, and
  `avex` (BEATs-bio) publishes no wheel for 3.14. 3.12 satisfies everything.

### What you'll get
1. Each vocalization clip as an in-App **spectrogram** tile you can browse and **play**.
2. A 2-D **embedding plot** for every model, coloured by species / family / order.
3. **Similarity search** — via the toolkit's panel *and* from the notebook.
4. A side-by-side view where you toggle models and watch clusters reorganise by clade.

### What this demo does *not* do
It does not reproduce the Mantel-test statistics (that's the paper's actual
contribution). It shows the **qualitative** signal those statistics measure. Sample
counts are small by ML standards (52 species), so clusters look clean but sparse.

---

## 0. Start clean — stop stale processes (terminal)

Do this **in a terminal**, before anything else. FiftyOne's App server, its
delegated-operation workers, and its bundled MongoDB can outlive a crashed kernel and hold
onto ports/locks. Clearing them first avoids confusing "port in use" or stale-database
errors.

**macOS / Linux:**
```bash
# 1. See what FiftyOne / Mongo / Jupyter processes are running
ps aux | grep -iE 'fiftyone|mongod|jupyter' | grep -v grep

# 2. Stop them (order: app server -> db service -> mongod)
pkill -f "fiftyone.server.main"    || true   # App server
pkill -f "fiftyone.service"        || true   # service wrapper (supervises mongod)
pkill -f "fiftyone/db/bin/mongod"  || true   # FiftyOne's bundled mongod
pkill -f "jupyter"                 || true   # notebook/lab servers + stray kernels

# FiftyOne's mongod uses an auto-assigned port, so don't blindly kill :27017
# (that could be a different MongoDB). Just free the App's default port:
lsof -ti :5151 | xargs kill -9  2>/dev/null || true

# 3. Re-check: you want NOTHING left
sleep 2
ps aux | grep -iE 'fiftyone|mongod|jupyter' | grep -v grep || echo "clean: nothing left"
```

**Windows (PowerShell):**
```powershell
# See any FiftyOne / Mongo / Jupyter processes
Get-Process | Where-Object { $_.ProcessName -match 'fiftyone|mongod|jupyter|python' } | Format-Table Id, ProcessName

# Stop them (adjust as needed)
Get-Process fiftyone*, mongod, jupyter* -ErrorAction SilentlyContinue | Stop-Process -Force
```

This is optional on a fresh machine that has never run FiftyOne — it matters mainly if a
previous session crashed. If FiftyOne isn't installed yet, these commands simply find
nothing, which is fine.

---

## 0b. Create the virtual environment (terminal)

Use a dedicated, throwaway virtual environment so nothing touches your other Python setups.

**Python version:** use **3.10, 3.11, or 3.12**. FiftyOne supports 3.10–3.13, but this
demo's audio dependencies are the tighter constraint — `librosa` (via `numba`/`llvmlite`)
and `avex` (the BEATs-bio package) lag on the newest Python releases, so **3.12 is the
safe, well-tested choice** and newer versions (3.13+) may fail to install those wheels. If
your system `python3` is newer than 3.12, install 3.12 alongside it:
- **macOS:** `brew install python@3.12`
- **Ubuntu/Debian:** `sudo apt install python3.12 python3.12-venv`
- **Windows / any OS:** download from [python.org](https://www.python.org/downloads/) or use
  `conda create -n fiftyone-demo python=3.12`

```bash
# --- 1. clone the source repo (embeddings + metadata are committed) ---
git clone https://github.com/rinvictor/bioacoustic-phylogeny-embeddings.git
cd bioacoustic-phylogeny-embeddings

# --- 2. create a venv with a 3.10-3.12 interpreter ---
#     Replace `python3.12` with whatever resolves to a supported version on your machine.
python3.12 -m venv .venv-fiftyone-demo

#     Activate it:
#       macOS / Linux:
source .venv-fiftyone-demo/bin/activate
#       Windows (PowerShell):
#     .venv-fiftyone-demo\Scripts\Activate.ps1

python -m pip install --upgrade pip setuptools wheel
python --version                 # confirm it prints 3.10.x / 3.11.x / 3.12.x

# --- 3. install open-source FiftyOne + the demo dependencies ---
pip install fiftyone
pip install librosa matplotlib numpy pandas soundfile ipykernel umap-learn
pip install datasets huggingface_hub    # for the audio download
pip install transformers torch          # for CLAP + AST embeddings
pip install avex                         # BEATs-bio embeddings
pip install requests                     # used by the toolkit's operators

# --- 4. sanity-check the environment ---
python - <<'PY'
import sys, fiftyone as fo
print("python  :", sys.version.split()[0])
print("fiftyone:", fo.__version__)
assert (3, 10) <= sys.version_info[:2] <= (3, 12), \
    "Use Python 3.10-3.12 for this demo's audio stack."
PY

# --- 5. register the venv as a Jupyter kernel ---
python -m ipykernel install --user --name fiftyone-demo --display-name "Python (fiftyone-demo)"
```

> **Tip:** if `avex` or `librosa` fail to install, you are almost certainly on a Python
> version newer than 3.12. Recreate the venv with a 3.10–3.12 interpreter and try again.

## 0c. Install the audio toolkit plugin (terminal)

The in-App spectrograms + audio player come from the community
[`@roboav8r/fiftyone-audio-toolkit`](https://github.com/roboav8r/fiftyone-audio-toolkit)
plugin — no Node, no build step. One small fix is needed: the plugin's manifest declares a
FiftyOne *Enterprise* version (`">=2.18.0"`), which won't validate against open-source
FiftyOne even though the features it needs are present. We relax that pin to the
open-source line. The patch below is written in Python so it works identically on macOS,
Linux, and Windows (no `sed` needed).

```bash
# --- 1. download the toolkit into your plugins dir ---
fiftyone plugins download https://github.com/roboav8r/fiftyone-audio-toolkit
```

```bash
# --- 2. relax the Enterprise version pin -> open-source line (cross-platform) ---
python - <<'PY'
import glob, os, re, fiftyone as fo

plugins_dir = fo.config.plugins_dir
# find the toolkit's manifest wherever it landed (it installs under a @roboav8r/ dir)
hits = glob.glob(os.path.join(plugins_dir, "**", "fiftyone.yml"), recursive=True)
target = None
for h in hits:
    txt = open(h).read()
    if "fiftyone-audio-toolkit" in txt or "audio_embeddings_search" in txt:
        target = h; break

if not target:
    raise SystemExit("Could not find the toolkit's fiftyone.yml under %s" % plugins_dir)

src = open(target).read()
patched = re.sub(r'version:\s*">=2\.18\.0"', 'version: ">=1.15"', src)
if patched != src:
    open(target + ".bak", "w").write(src)      # backup
    open(target, "w").write(patched)
    print("patched:", target)
else:
    print("no change needed (already patched?):", target)
PY
```

```bash
# --- 3. confirm FiftyOne sees it, enabled ---
fiftyone plugins list
#   expect a row:  @roboav8r/fiftyone-audio-toolkit  0.2.0  ...  (enabled)
```

Then fetch the audio and compute the embeddings so they align with the committed metadata:

```bash
# --- 4. download the Watkins marine-mammal clips into mammals/data/ (~hundreds of MB) ---
python mammals/download.py

# --- 5. compute the embeddings, in metadata.csv row order ---
python mammals/embed.py            # CLAP      -> clap_embeddings.npy
python mammals/embed_ast.py        # AST       -> ast_embeddings.npy
python mammals/embed_beats_bio.py  # BEATs-bio -> beats_bio_embeddings.npy
# (The MFCC baseline is generated from this notebook, in cell 1b.)
#
# NOTE: these scripts download model weights on first run and process ~1,700 clips,
# so they take a while and need network access.

# --- 6. confirm every array's row count matches metadata.csv ---
python - <<'PY'
import numpy as np, glob, os
for f in sorted(glob.glob("mammals/embeddings/*.npy")):
    print(os.path.basename(f), np.load(f).shape)
PY

# --- 7. launch Jupyter and open THIS notebook, selecting the "Python (fiftyone-demo)" kernel ---
pip install jupyterlab
jupyter lab
```

> The repo ships committed `.npy` embeddings, but they may be from a different run/order
> than the current `metadata.csv`. Recomputing (step 5) guarantees the embeddings line up
> with the metadata row-for-row, which this notebook relies on. If you prefer to trust the
> committed arrays, you can skip step 5 — the notebook's cell 4 will warn you if any array
> doesn't match the metadata row count.

---

## 1. Configuration

Set `REPO_ROOT` to wherever you cloned the repo. If you launched Jupyter from inside the
repo (per cell 0c), the default `"."` is correct.

`TAXON` picks which radiation to load: `"mammals"`, `"birds"`, or `"both"`.

`MAX_CLIPS_PER_SPECIES` caps how many clips per species get loaded into the App. There's no
slow spectrogram-render step anymore (the toolkit draws them in-browser), so this is just a
"how busy is the grid" knob. Start small (e.g. 8) then raise it.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path(".").resolve()          # <-- edit if needed
TAXON = "mammals"                         # "mammals" | "birds" | "both"
MAX_CLIPS_PER_SPECIES = 8                 # None = no cap (loads every clip)
DATASET_NAME = "bioacoustic-phylogeny"

# The audio toolkit provides in-App spectrograms + playback. We verify it's installed so
# the App renders audio tiles instead of "unsupported media".
AUDIO_TOOLKIT = "@roboav8r/fiftyone-audio-toolkit"

# Force a specific metadata CSV per taxon if auto-selection picks the wrong one.
METADATA_CSV = {
    # "mammals": "meta/metadata.csv",
}

assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT}"

# --- environment sanity: open-source FiftyOne on Python 3.12 ---
import fiftyone as fo
print("Python    :", sys.version.split()[0], "(supported: 3.10-3.12 for this demo)")
print("FiftyOne  :", fo.__version__, "(open-source 1.20.x line)")
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    print("  WARNING: outside Python 3.10-3.12 -- the audio deps (numba/librosa, avex) may fail to install.")

# --- confirm the audio toolkit is installed & enabled ---
try:
    import fiftyone.plugins as fop
    names = {p.name for p in fop.list_plugins()}
    if AUDIO_TOOLKIT in names:
        print(f"audio toolkit  : installed & enabled ({AUDIO_TOOLKIT})")
    else:
        print(f"audio toolkit  : NOT FOUND -- in-App spectrograms/playback will be missing.")
        print("  install it (see cell 0c):")
        print("    fiftyone plugins download https://github.com/roboav8r/fiftyone-audio-toolkit")
        print('    then relax the manifest version pin ">=2.18.0" -> ">=1.15".')
except Exception as e:
    print("(could not list plugins to verify the toolkit:", e, ")")

print("Repo root :", REPO_ROOT)
print("Taxon     :", TAXON)


## 1b. (Optional) Generate the MFCC baseline

The paper's cleanest contrast is that hand-crafted **MFCC** features recover almost no
phylogenetic signal (Mantel r≈0.04) while the learned foundation-model embeddings recover a
lot. To show that contrast you need an `mfcc_embeddings.npy`.

This cell reproduces `embed.py`'s **exact 105-dim hand-crafted feature recipe** directly,
reading `metadata.csv` in row order — so the result aligns to the CSV by construction. It
writes the file only if it doesn't already exist or you set `FORCE_MFCC = True`. Set
`RUN_MFCC = False` to skip. Requires the audio to be downloaded; runs on CPU in ~a minute.

In [ ]:
RUN_MFCC   = True     # set False to skip MFCC generation
FORCE_MFCC = False    # set True to overwrite an existing mfcc_embeddings.npy
MFCC_SR    = 22050    # matches embed.py's SR

if RUN_MFCC:
    import numpy as np, pandas as pd, librosa, soundfile as sf
    from pathlib import Path

    mbase = REPO_ROOT / "mammals"
    out = mbase / "embeddings" / "mfcc_embeddings.npy"
    if out.exists() and not FORCE_MFCC:
        arr = np.load(out)
        print(f"[mfcc] {out.name} already exists: shape {arr.shape} "
              f"(set FORCE_MFCC=True to overwrite)")
    else:
        csv = mbase / "meta" / "metadata.csv"
        rows = pd.read_csv(csv)
        print(f"[mfcc] computing 105-dim hand-crafted features for {len(rows)} clips "
              f"(40 MFCC mean+std, chroma, spectral contrast, centroid, rolloff, ZCR)...")

        def features_105(audio, sr):
            mfcc     = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            chroma   = librosa.feature.chroma_stft(y=audio, sr=sr)
            contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
            centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
            rolloff  = librosa.feature.spectral_rolloff(y=audio, sr=sr)
            zcr      = librosa.feature.zero_crossing_rate(audio)
            return np.concatenate([
                mfcc.mean(axis=1), mfcc.std(axis=1),
                chroma.mean(axis=1),
                contrast.mean(axis=1),
                centroid.mean(axis=1), centroid.std(axis=1),
                rolloff.mean(axis=1), rolloff.std(axis=1),
                zcr.mean(axis=1), zcr.std(axis=1),
            ])

        feats, n_fail = [], 0
        for i, r in enumerate(rows["file"]):
            wav = mbase / str(r)
            try:
                audio, sr = sf.read(str(wav), dtype="float32")
                if audio.ndim > 1:
                    audio = audio.mean(axis=1)
                if sr != MFCC_SR:
                    audio = librosa.resample(audio, orig_sr=sr, target_sr=MFCC_SR)
                vec = features_105(audio, MFCC_SR)
            except Exception as ex:
                n_fail += 1
                vec = np.zeros(105, dtype=np.float32)
                if n_fail <= 5:
                    print(f"   [mfcc] failed on row {i} ({wav.name}): {ex}")
            feats.append(np.asarray(vec, dtype=np.float32))
            if (i + 1) % 200 == 0:
                print(f"   {i+1}/{len(rows)}")

        X = np.vstack(feats)
        assert X.shape[1] == 105, f"expected 105 dims, got {X.shape[1]}"
        np.save(str(out), X)
        print(f"[mfcc] wrote {out}  shape={X.shape}  ({n_fail} clips failed -> zero vectors)")
else:
    print("[mfcc] skipped (RUN_MFCC = False)")


## 2. Discover what the repo actually ships

Scan the taxon folder for its metadata CSV, its `embeddings/*.npy` arrays, and its raw
audio, printing what it finds so you can spot mismatches. Raw audio matters here: the `.wav`
becomes each sample's media, so if audio didn't download you get placeholder tiles instead
of real, playable spectrograms.

In [ ]:
import numpy as np, pandas as pd

def discover(taxon):
    base = REPO_ROOT / taxon
    info = {"taxon": taxon, "base": base}

    emb_dir = base / "embeddings"
    emb_rows = None
    if emb_dir.exists():
        for f in emb_dir.glob("*.npy"):
            try:
                a = np.load(f, allow_pickle=True)
                if getattr(a, "ndim", 0) == 2:
                    emb_rows = a.shape[0]; break
            except Exception:
                pass

    csv = None
    pin = METADATA_CSV.get(taxon) if "METADATA_CSV" in globals() else None
    if pin:
        pin = Path(pin)
        csv = pin if pin.is_absolute() else (base / pin)
    if csv is None and base.exists():
        cands = sorted(base.rglob("*.csv"))
        def rows(p):
            try: return len(pd.read_csv(p))
            except Exception: return -1
        if emb_rows is not None:
            matches = [c for c in cands if rows(c) == emb_rows]
            if matches:
                meta_m = [c for c in matches if "meta" in c.name.lower()]
                csv = (meta_m or matches)[0]
        if csv is None:
            direct = base / "metadata.csv"
            meta_named = [c for c in cands if "meta" in c.name.lower()]
            csv = direct if direct.exists() else (meta_named or cands or [None])[0]

    if csv is not None and Path(csv).exists():
        df = pd.read_csv(csv)
        info["metadata"] = df
        tag = ""
        if emb_rows is not None:
            tag = "  [MATCHES embeddings]" if len(df) == emb_rows else f"  [!! {len(df)} rows vs {emb_rows} embedding rows]"
        print(f"[{taxon}] metadata CSV: {Path(csv).relative_to(REPO_ROOT)}  ->  {len(df)} rows{tag}")
        print("    columns:", list(df.columns))
        if emb_rows is not None and len(df) != emb_rows:
            print(f"    other CSVs found (set METADATA_CSV['{taxon}'] to force one):")
            for c in sorted(base.rglob('*.csv')):
                try: rc = len(pd.read_csv(c))
                except Exception: rc = '?'
                print(f"        {c.relative_to(base)}  ({rc} rows)")
    else:
        info["metadata"] = None
        print(f"[{taxon}] NO csv found under {base}")

    embs = {}
    if emb_dir.exists():
        for f in sorted(emb_dir.glob("*.npy")):
            arr = np.load(f, allow_pickle=True)
            embs[f.stem] = arr
            print(f"[{taxon}] embeddings/{f.name:<24} shape={getattr(arr,'shape',None)} dtype={arr.dtype}")
    else:
        print(f"[{taxon}] NO embeddings/ dir at {emb_dir}")
    info["embeddings"] = embs

    data_dir = base / "data"
    n_audio = 0
    if data_dir.exists():
        n_audio = sum(1 for _ in data_dir.rglob("*.wav")) + sum(1 for _ in data_dir.rglob("*.flac"))
    info["data_dir"] = data_dir if data_dir.exists() else None
    info["n_audio"] = n_audio
    print(f"[{taxon}] raw audio files found: {n_audio}"
          + ("" if n_audio else "  (run download scripts -- without audio you get placeholder tiles, no spectrogram/playback)"))
    print()
    return info

taxa = ["mammals", "birds"] if TAXON == "both" else [TAXON]
discovered = {t: discover(t) for t in taxa}


## 3. Map the metadata to standard fields

Different commits name columns differently (`species` vs `label`, `file` vs `path`, etc.).
This cell guesses the right column for each concept and prints its choices so you can
override any that look wrong via `OVERRIDES`, then re-run.

In [ ]:
CANDIDATES = {
    "species": ["species", "label", "species_name", "common_name", "scientific_name", "class"],
    "family":  ["family", "family_name"],
    "order":   ["order", "order_name", "clade"],
    "path":    ["path", "filepath", "file", "filename", "audio", "audio_path", "clip", "wav"],
    "id":      ["id", "clip_id", "uid", "index", "recording_id"],
    "freq":    ["dominant_frequency", "dom_freq", "freq", "peak_freq"],
}

OVERRIDES = {   # e.g. {"mammals": {"species": "label"}}
    "mammals": {"species": "species_label"},
}

def resolve_columns(df, taxon):
    cols = {c.lower(): c for c in df.columns}
    chosen = {}
    for concept, cands in CANDIDATES.items():
        pick = None
        for cand in cands:
            if cand.lower() in cols:
                pick = cols[cand.lower()]; break
        chosen[concept] = pick
    chosen.update(OVERRIDES.get(taxon, {}))
    return chosen

colmap = {}
for t in taxa:
    df = discovered[t]["metadata"]
    if df is None:
        print(f"[{t}] no metadata; skipping"); continue
    cm = resolve_columns(df, t)
    colmap[t] = cm
    print(f"[{t}] column mapping:")
    for k, v in cm.items():
        flag = "" if v else "   <-- not found"
        print(f"    {k:<8} -> {v}{flag}")
    if not cm["species"]:
        print(f"    WARNING: no species column resolved for {t}; set OVERRIDES and re-run.")
    print()


## 4. Resolve audio paths & align rows with embedding rows

Two things must line up: the *i*-th row of `metadata.csv` and the *i*-th row of each
`embeddings/*.npy`. The repo builds them in the same order, so we keep only the embedding
arrays whose length matches, and resolve each clip to an audio file on disk — which the
toolkit needs, since the resolved `.wav` becomes the sample's media (and thus its in-App
spectrogram + player).

In [ ]:
def resolve_audio_path(raw, taxon):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    raw = str(raw)
    base = discovered[taxon]["base"]
    cands = [Path(raw), base / raw, base / "data" / raw, base / "data" / Path(raw).name]
    for c in cands:
        if c.exists():
            return c.resolve()
    data_dir = discovered[taxon].get("data_dir")
    if data_dir:
        hits = list(data_dir.rglob(Path(raw).name))
        if hits:
            return hits[0].resolve()
    return None

aligned = {}
for t in taxa:
    df = discovered[t]["metadata"]
    embs = discovered[t]["embeddings"]
    if df is None or not embs:
        print(f"[{t}] missing metadata or embeddings; skipping"); continue

    n_meta = len(df)
    usable = {}
    for name, arr in embs.items():
        a = np.asarray(arr)
        if a.ndim == 2 and a.shape[0] == n_meta:
            usable[name] = a
        else:
            print(f"[{t}] skipping embedding '{name}' (shape {a.shape} != ({n_meta}, d))")
    if not usable:
        print(f"[{t}] no embedding array aligns with {n_meta} metadata rows; skipping"); continue

    cm = colmap[t]
    df = df.copy()
    df["_species"] = df[cm["species"]].astype(str) if cm["species"] else "unknown"
    df["_family"]  = df[cm["family"]].astype(str)  if cm["family"]  else None
    df["_order"]   = df[cm["order"]].astype(str)   if cm["order"]   else None
    df["_freq"]    = df[cm["freq"]]                 if cm["freq"]    else None
    raw_paths      = df[cm["path"]] if cm["path"] else [None] * n_meta
    df["_audio"]   = [resolve_audio_path(p, t) for p in raw_paths]

    n_resolved = df["_audio"].notna().sum()
    print(f"[{t}] {n_meta} rows | models: {list(usable)} | audio resolved: {n_resolved}/{n_meta}")
    if n_resolved == 0:
        print(f"    NOTE: no audio resolved -> tiles will be placeholders (no spectrogram/playback). "
              f"Run mammals/download.py.")
    aligned[t] = {"df": df, "embeddings": usable}
print("\nAlignment complete.")


## 4b. Attach marine-mammal taxonomy (clade / order / family)

The repo's `metadata.csv` has species but no higher taxonomy, so we supply a **verified**
mapping for the 32 Watkins species. The split that matters is **cetacean vs pinniped**:
within the 26 cetaceans the phylogenetic signal is strong (CLAP r≈0.82), and the deep
cetacean–pinniped divergence is what dilutes the full 32-species correlation. Colouring the
embedding plot by `clade` (and `family`) is what makes that visible. Unmatched species get
`clade="unknown"` and are reported.

In [ ]:
MAMMAL_TAXONOMY = {
    "Bowhead_Whale":                 ("Cetacean", "Cetacea", "Balaenidae"),
    "Northern_Right_Whale":          ("Cetacean", "Cetacea", "Balaenidae"),
    "Southern_Right_Whale":          ("Cetacean", "Cetacea", "Balaenidae"),
    "Humpback_Whale":                ("Cetacean", "Cetacea", "Balaenopteridae"),
    "Fin,_Finback_Whale":            ("Cetacean", "Cetacea", "Balaenopteridae"),
    "Minke_Whale":                   ("Cetacean", "Cetacea", "Balaenopteridae"),
    "Sperm_Whale":                   ("Cetacean", "Cetacea", "Physeteridae"),
    "Beluga,_White_Whale":           ("Cetacean", "Cetacea", "Monodontidae"),
    "Narwhal":                       ("Cetacean", "Cetacea", "Monodontidae"),
    "Atlantic_Spotted_Dolphin":      ("Cetacean", "Cetacea", "Delphinidae"),
    "Pantropical_Spotted_Dolphin":   ("Cetacean", "Cetacea", "Delphinidae"),
    "Bottlenose_Dolphin":            ("Cetacean", "Cetacea", "Delphinidae"),
    "Clymene_Dolphin":               ("Cetacean", "Cetacea", "Delphinidae"),
    "Common_Dolphin":                ("Cetacean", "Cetacea", "Delphinidae"),
    "False_Killer_Whale":            ("Cetacean", "Cetacea", "Delphinidae"),
    "Frasers_Dolphin":               ("Cetacean", "Cetacea", "Delphinidae"),
    "Grampus,_Rissos_Dolphin":       ("Cetacean", "Cetacea", "Delphinidae"),
    "Killer_Whale":                  ("Cetacean", "Cetacea", "Delphinidae"),
    "Long-Finned_Pilot_Whale":       ("Cetacean", "Cetacea", "Delphinidae"),
    "Short-Finned_Pacific_Pilot_Whale": ("Cetacean", "Cetacea", "Delphinidae"),
    "Melon_Headed_Whale":            ("Cetacean", "Cetacea", "Delphinidae"),
    "Rough-Toothed_Dolphin":         ("Cetacean", "Cetacea", "Delphinidae"),
    "Spinner_Dolphin":               ("Cetacean", "Cetacea", "Delphinidae"),
    "Striped_Dolphin":               ("Cetacean", "Cetacea", "Delphinidae"),
    "White-beaked_Dolphin":          ("Cetacean", "Cetacea", "Delphinidae"),
    "White-sided_Dolphin":           ("Cetacean", "Cetacea", "Delphinidae"),
    "Bearded_Seal":                  ("Pinniped", "Carnivora", "Phocidae"),
    "Harp_Seal":                     ("Pinniped", "Carnivora", "Phocidae"),
    "Leopard_Seal":                  ("Pinniped", "Carnivora", "Phocidae"),
    "Ross_Seal":                     ("Pinniped", "Carnivora", "Phocidae"),
    "Weddell_Seal":                  ("Pinniped", "Carnivora", "Phocidae"),
    "Walrus":                        ("Pinniped", "Carnivora", "Odobenidae"),
}

if "mammals" in aligned:
    df = aligned["mammals"]["df"]
    clade  = df["_species"].map(lambda s: MAMMAL_TAXONOMY.get(s, ("unknown","unknown","unknown"))[0])
    order  = df["_species"].map(lambda s: MAMMAL_TAXONOMY.get(s, ("unknown","unknown","unknown"))[1])
    family = df["_species"].map(lambda s: MAMMAL_TAXONOMY.get(s, ("unknown","unknown","unknown"))[2])
    df["_clade"], df["_order"], df["_family"] = clade, order, family

    unmatched = sorted(set(df.loc[df["_clade"]=="unknown", "_species"]))
    n_cet = int((df["_clade"]=="Cetacean").sum())
    n_pin = int((df["_clade"]=="Pinniped").sum())
    print(f"[mammals] taxonomy attached: {n_cet} cetacean clips, {n_pin} pinniped clips.")
    if unmatched:
        print(f"[mammals] unmatched species (clade=unknown): {unmatched}")
    else:
        print("[mammals] all species matched the taxonomy table.")
else:
    print("No mammals in `aligned` -- skipping mammal taxonomy.")


## 5. Subsample (optional) and choose the sample media

No slow spectrogram-render step here — the audio toolkit draws spectrograms in-browser. This
cell just (a) records each row's true positional index (`_row`, which maps a row to its
embedding vector), (b) optionally caps clips per species, and (c) chooses the media path for
each sample: the resolved **`.wav`** when we have it (so the renderer + player take over),
else a one-time placeholder PNG so the embedding plots still work.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Placeholders only appear when a clip's audio didn't resolve; real clips render as
# spectrograms in-browser by @roboav8r/fiftyone-audio-toolkit.
PLACEHOLDER_DIR = REPO_ROOT / "_demo_placeholders"
PLACEHOLDER_DIR.mkdir(exist_ok=True)

def placeholder_tile(text, out_png):
    fig, ax = plt.subplots(figsize=(3.2, 2.2), dpi=100)
    ax.text(0.5, 0.5, text, ha="center", va="center", wrap=True, fontsize=8)
    ax.set_axis_off()
    fig.savefig(out_png, bbox_inches="tight", pad_inches=0.1)
    plt.close(fig)

for t in taxa:
    if t not in aligned:
        continue
    df = aligned[t]["df"].copy()

    # TRUE positional row index BEFORE any sampling -> maps to embs[m][_row]
    df = df.reset_index(drop=True)
    df["_row"] = np.arange(len(df))

    if MAX_CLIPS_PER_SPECIES:
        keep = []
        for sp, idx in df.groupby("_species", sort=False).groups.items():
            keep.extend(list(idx)[:MAX_CLIPS_PER_SPECIES])
        df = df.loc[keep].reset_index(drop=True)

    # Choose each sample's media: the .wav when resolved, else a placeholder PNG.
    n_audio, n_place = 0, 0
    media_paths = []
    for _, r in df.iterrows():
        ap = r["_audio"]
        if ap is not None:
            media_paths.append(str(ap)); n_audio += 1
        else:
            png = PLACEHOLDER_DIR / t / f"{t}_{int(r['_row']):05d}.png"
            png.parent.mkdir(parents=True, exist_ok=True)
            if not png.exists():
                placeholder_tile(f"{r['_species']}\n(row {int(r['_row'])})\nno audio", png)
            media_paths.append(str(png)); n_place += 1
    df["_media"] = media_paths

    aligned[t]["df_sub"] = df
    print(f"[{t}] {len(df)} samples selected -> media: {n_audio} audio (.wav), {n_place} placeholder")


## 6. Build the FiftyOne dataset

Each sample's media is the **`.wav` itself** (falling back to a placeholder PNG only when
audio didn't resolve). Because audio reports media type `unknown`, the toolkit's renderer
takes over those tiles automatically — spectrogram in the grid and modal, `<audio>` player +
playhead in the modal. We also attach taxonomy fields and every model's embedding vector, so
we can feed them to `compute_visualization` and the similarity operators.

In [ ]:
import fiftyone as fo

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME, persistent=False)

all_models = sorted({m for t in aligned for m in aligned[t]["embeddings"]})
print("Embedding models:", all_models)

samples = []
for t in taxa:
    if t not in aligned:
        continue
    if "df_sub" not in aligned[t]:
        raise RuntimeError(
            f"[{t}] 'df_sub' not found -- run cell 5 before this cell. "
            "Best fix: Kernel -> Restart & Run All.")
    df = aligned[t]["df_sub"]
    required = ["_row", "_species", "_media"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"[{t}] dataframe missing {missing}; run cells 4, 4b, 5 in order.")
    embs = aligned[t]["embeddings"]
    for _, r in df.iterrows():
        row = int(r["_row"])
        # filepath = the .wav (rendered as a spectrogram + player by the toolkit) or a placeholder
        s = fo.Sample(filepath=r["_media"])
        s["taxon"]   = t
        s["species"] = r["_species"]
        for fld, col in [("clade","_clade"), ("family","_family"), ("order","_order")]:
            v = r.get(col)
            if v is not None and pd.notna(v):
                s[fld] = str(v)
        # keep an explicit audio_path too (handy for bookkeeping / optional notebook playback)
        if r.get("_audio") is not None:
            s["audio_path"] = str(r["_audio"])
        if r.get("_freq") is not None and pd.notna(r["_freq"]):
            try: s["dominant_freq"] = float(r["_freq"])
            except Exception: pass
        for m in all_models:
            if m in embs:
                s[f"emb_{m}"] = np.asarray(embs[m][row], dtype=float).tolist()
        samples.append(s)

dataset.add_samples(samples)
print(f"\nAdded {len(dataset)} samples.")
print("Species :", len(set(dataset.values("species"))))
n_wav = sum(1 for fp in dataset.values("filepath")
            if str(fp).lower().endswith((".wav",".flac",".mp3",".ogg",".aac",".m4a",".opus")))
print(f"Audio-media samples (get in-App spectrogram + player): {n_wav}/{len(dataset)}")
print("Fields  :", [f for f in dataset.get_field_schema()
                    if f.startswith(('emb_','species','clade','family','order','taxon'))])


## 7. Compute an embedding visualization for every model

For each model we run UMAP (falls back to PCA if `umap-learn` isn't installed) on its
precomputed embeddings and store the 2-D result under a distinct `brain_key`. In the App
you'll switch between these keys in the Embeddings panel to compare how each model organises
the same clips.

In [ ]:
import fiftyone.brain as fob

def embeddings_matrix(model):
    field = f"emb_{model}"
    ids, vecs = [], []
    for sid, v in zip(dataset.values("id"), dataset.values(field)):
        if v is not None:
            ids.append(sid); vecs.append(v)
    return ids, np.asarray(vecs, dtype=float)

method = "umap"
try:
    import umap  # noqa
except Exception:
    method = "pca"
    print("umap-learn not installed -> using PCA (pip install umap-learn for nicer clusters).")

for m in all_models:
    ids, X = embeddings_matrix(m)
    if len(X) < 3:
        print(f"[{m}] too few vectors ({len(X)}); skipping"); continue
    key = f"viz_{m}".replace("-", "_")
    if key in dataset.list_brain_runs():
        dataset.delete_brain_run(key)
    fob.compute_visualization(dataset, embeddings=X, brain_key=key,
                              method=method, seed=51, progress=False)
    print(f"[{m}] visualization stored under brain_key='{key}'  ({len(X)} pts)")

print("\nAll visualizations:", [k for k in dataset.list_brain_runs() if k.startswith('viz_')])


## 8. Similarity search — "what sounds like this?"

Two complementary paths:

- **In the App**, the toolkit's **Audio Embeddings Search** panel computes a query embedding
  from a selected sample, an uploaded clip, or free text (local CLAP / MS-CLAP / an
  endpoint), then lets you drag a distance threshold and read a class histogram. That's the
  interactive path — cell 9 shows where to find it.
- **Here in the notebook**, we build a `compute_similarity` index directly on the precomputed
  embeddings so you can sort the whole dataset by proximity to any clip programmatically —
  the same index type the App's "sort by similarity" uses.

This cell does the notebook one.

In [ ]:
sim_model = "clap" if any("clap" in m.lower() for m in all_models) else all_models[0]
sim_model = next(m for m in all_models if sim_model.lower() in m.lower())

ids, X = embeddings_matrix(sim_model)
sim_key = f"sim_{sim_model}".replace("-", "_")
if sim_key in dataset.list_brain_runs():
    dataset.delete_brain_run(sim_key)

fob.compute_similarity(dataset, embeddings=X, brain_key=sim_key, backend="sklearn")
print(f"Similarity index '{sim_key}' built on {sim_model} ({len(X)} vectors).")

query_id = dataset.first().id
view = dataset.sort_by_similarity(query_id, k=10, brain_key=sim_key)
print(f"\nQuery clip species: {dataset[query_id].species}")
print("Top-10 acoustic neighbours (species):")
for s in view:
    print("   ", s.species)


## 8b. Save the good demo views

Register **saved views** so the compelling demo states are one click away in the App's view
dropdown (top-left). You pick the Embeddings-panel brain key and "color by" in the App, so
each view's description says which to choose. Re-running is safe.

In [ ]:
from fiftyone import ViewField as F

def save_view(name, view, description):
    if name in dataset.list_saved_views():
        dataset.delete_saved_view(name)
    dataset.save_view(name, view)
    try:
        dataset.update_saved_view_info(name, dict(description=description))
    except Exception:
        pass
    print(f"  saved view: {name:28s} ({len(view)} samples)")

print("Registering saved views...")
save_view("All clips", dataset.view(),
          "Full dataset. Embeddings panel: try each viz_* brain key, colour by clade.")
save_view("Cetaceans only", dataset.match(F("clade") == "Cetacean"),
          "26-species cetacean clade (strongest signal). Colour by FAMILY, brain key "
          "viz_clap -- delphinids vs baleen whales vs monodontids pull apart.")
save_view("Pinnipeds only", dataset.match(F("clade") == "Pinniped"),
          "6 pinniped species (seals + walrus). Contrast against the cetaceans.")

family_views = {
    "Delphinidae (dolphins)": "Delphinidae",
    "Baleen whales (Balaenidae)": "Balaenidae",
    "Baleen whales (Balaenopteridae)": "Balaenopteridae",
    "Monodontidae (beluga+narwhal)": "Monodontidae",
}
present_families = set(dataset.distinct("family"))
for label, fam in family_views.items():
    if fam in present_families:
        save_view(label, dataset.match(F("family") == fam),
                  f"Single family: {fam}. Colour by species to see within-family spread.")

save_view("Model comparison (cetaceans)", dataset.match(F("clade") == "Cetacean"),
          "Same cetacean slice -- flip brain key across viz_clap / viz_ast / viz_beats_bio "
          "(colour by family). Specialist BEATs-bio shouldn't beat the generalists.")

print("\nAll saved views:")
for v in dataset.list_saved_views():
    print("  -", v)


## 8c. Per-species mean embeddings — the low-noise view

The per-clip plots are visually rich but noisy. The **Mantel test in the paper doesn't work
on clips at all** — it collapses each species to a single mean embedding and correlates the
32×32 acoustic distance matrix against phylogeny. This cell builds exactly that view: one
point per species (averaged over all its clips), in a **separate** dataset. Each species
point uses its **medoid clip** (the real clip closest to the species mean) as its media — so
with the toolkit installed those tiles are still real, playable spectrograms.

In [ ]:
import fiftyone as fo, fiftyone.brain as fob, numpy as np

SPECIES_DATASET = f"{DATASET_NAME}-species"

model_fields = [f for f in dataset.get_field_schema() if f.startswith("emb_")]
if not model_fields:
    raise RuntimeError("No emb_* fields on the dataset -- run cells up through 6 first.")

ids       = dataset.values("id")
species   = dataset.values("species")
clades    = dataset.values("clade")
families  = dataset.values("family")
orders    = dataset.values("order")
media     = dataset.values("filepath")   # the .wav (or placeholder) -- used as the tile
audio     = dataset.values("audio_path") if "audio_path" in dataset.get_field_schema() else [None]*len(ids)
emb_cols  = {m: np.asarray(dataset.values(m), dtype=object) for m in model_fields}

from collections import defaultdict
rows_by_species = defaultdict(list)
for i, sp in enumerate(species):
    rows_by_species[sp].append(i)

if fo.dataset_exists(SPECIES_DATASET):
    fo.delete_dataset(SPECIES_DATASET)
sds = fo.Dataset(SPECIES_DATASET, persistent=False)

species_order = sorted(rows_by_species)
mean_vectors  = {m: [] for m in model_fields}
samples = []
for sp in species_order:
    idxs = rows_by_species[sp]
    primary = next((m for m in model_fields if "clap" in m.lower()), model_fields[0])
    M = np.vstack([np.asarray(emb_cols[primary][i], dtype=float) for i in idxs])
    centroid = M.mean(axis=0)
    medoid_local = int(np.argmin(np.linalg.norm(M - centroid, axis=1)))
    medoid_i = idxs[medoid_local]

    # medoid clip's own media (.wav) -> real spectrogram + player via the toolkit
    s = fo.Sample(filepath=media[medoid_i])
    s["species"] = sp
    s["clade"]   = clades[medoid_i]
    s["family"]  = families[medoid_i]
    s["order"]   = orders[medoid_i]
    s["n_clips"] = len(idxs)
    if audio[medoid_i] is not None:
        s["audio_path"] = audio[medoid_i]
    samples.append(s)

    for m in model_fields:
        Mm = np.vstack([np.asarray(emb_cols[m][i], dtype=float) for i in idxs])
        mean_vectors[m].append(Mm.mean(axis=0))

sds.add_samples(samples)
print(f"Built '{SPECIES_DATASET}': {len(sds)} species points.")

method = "umap"
try:
    import umap  # noqa
except Exception:
    method = "pca"
    print("umap-learn not installed -> PCA layout for species means.")

for m in model_fields:
    X = np.vstack(mean_vectors[m])
    key = f"viz_{m}".replace("-", "_")
    if key in sds.list_brain_runs():
        sds.delete_brain_run(key)
    kwargs = {}
    if method == "umap":
        kwargs["n_neighbors"] = min(15, len(X) - 1)
    fob.compute_visualization(sds, embeddings=X, brain_key=key, method=method,
                              seed=51, progress=False, **kwargs)
    print(f"  {m}: {key}  ({len(X)} species)")

from fiftyone import ViewField as F
for name, view in [
    ("Species means -- all", sds.view()),
    ("Species means -- cetaceans", sds.match(F("clade") == "Cetacean")),
]:
    if name in sds.list_saved_views():
        sds.delete_saved_view(name)
    sds.save_view(name, view)

print("\nLaunch this view with:  fo.launch_app(sds)")
print("Saved views on species dataset:", sds.list_saved_views())


## 8d. (Optional) Play a clip from the notebook

In-App playback via the toolkit is the main path (double-click a tile → modal player). But
if you want to audition a clip straight from a cell — e.g. while eyeballing the embedding
plot — this helper reads `audio_path` and renders a player inline. Works on any build.

In [ ]:
import IPython.display as ipd
from fiftyone import ViewField as F

def play_species(species_name, dataset=dataset):
    match = dataset.match(F("species") == species_name)
    if len(match) == 0:
        print(f"No samples for species '{species_name}'. Options:",
              sorted(set(dataset.values("species")))[:10], "...")
        return None
    s = match.first()
    ap = s.get_field("audio_path")
    if not ap:
        print(f"'{species_name}': sample has no audio_path (audio not downloaded?).")
        return None
    print(f"{species_name}  ({s.get_field('clade')} / {s.get_field('family')})  ->  {ap}")
    return ipd.Audio(ap)

# example: listen to a killer whale
play_species("Killer_Whale")


## 8e. Warm up the CLAP model (do this before using the search panel)

The toolkit's **Audio Embeddings Search** panel (in the App) can compute a query embedding
from a text prompt, an uploaded clip, or the selected sample using a **local CLAP** model.
That model's weights (`laion/larger_clap_music_and_speech`, ~600 MB) download from Hugging
Face on first use. If the very first thing you do is click *Compute query embedding* in the
panel before the weights are cached, the operator can fail on that cold first-load and the
panel shows an unhelpful `[object Object]` error.

Running this cell once pre-downloads and caches CLAP, so the panel works on the first click.
It's the same model the panel loads, so after this the App's local-CLAP backend loads it
instantly from cache. Safe to skip if you don't plan to use the panel's text/upload query.

In [ ]:
# Pre-cache CLAP so the App's "Audio Embeddings Search" panel works on first click.
# (Skips instantly on later runs once the weights are cached.)
WARM_UP_CLAP = True

if WARM_UP_CLAP:
    try:
        from transformers import ClapModel, ClapProcessor
        _m = ClapModel.from_pretrained("laion/larger_clap_music_and_speech")
        _p = ClapProcessor.from_pretrained("laion/larger_clap_music_and_speech")
        print("CLAP cached & loadable -> panel's Local CLAP backend will work on first use.")
        del _m, _p
    except Exception as e:
        print("Could not warm up CLAP:", e)
        print("The embedding plots + notebook similarity (cell 8) still work; only the "
              "App panel's local-CLAP query needs this.")
else:
    print("Skipped CLAP warm-up. If the App panel errors with [object Object] on first "
          "query, re-run this cell with WARM_UP_CLAP = True.")


## 9. Launch the app

Open the App, then in the **Embeddings** panel (the `+` next to *Samples*) pick a
`viz_<model>` brain key and colour by `clade`, `family`, or `species`. Toggle between models
and watch the clusters reorganise.

**Fastest path:** use the saved views from cell 8b (view dropdown, top-left). Load
**"Cetaceans only"**, open the Embeddings panel, set brain key `viz_clap_embeddings` and
colour by `family` — the strongest-looking view. Then flip the brain key across `viz_ast` and
`viz_beats_bio` to compare models on the same slice.

Reproduce the paper's punchline visually (marine mammals):
1. Colour by `clade`. Open `viz_clap` or `viz_ast` — cetaceans and pinnipeds separate at the
   clade level (a modest gradient; whole-matrix Mantel r≈0.41).
2. Load **"Cetaceans only"**, colour by `family` — the strong signal (r≈0.82): delphinids,
   baleen whales, and monodontids pull apart cleanly.
3. Flip to `viz_beats_bio` (specialist) on that slice — separation is comparable, not
   dramatically better. That *is* the finding. `viz_mfcc` shows the weakest structure.
4. Lasso a tight cluster → the Samples grid filters to those clips → **double-click** one to
   open the modal and **play it** (spectrogram + `<audio>` player + playhead), confirming
   they're the same family.

**Hearing the clips (in-App).** Just double-click any tile: the audio toolkit shows the
spectrogram and a native `<audio>` player with a playhead that tracks position across the
spectrogram — no server, no build. (For a quick inline audition without opening the App,
`play_species("Killer_Whale")` from cell 8d also works.)

**Similarity search (in-App).** Open the **Audio Embeddings Search** panel (the `+` next to
*Samples*) and query by a selected clip, an uploaded file, or free text, then drag the
distance threshold to explore the neighbourhood.

Note: `viz_birdnet` only exists if you load the **birds** taxon — BirdNET embeddings aren't
computed for marine mammals in this repo.

**Before using the search panel:** run cell 8e once to pre-cache CLAP, or the panel's first text/clip query may fail with an `[object Object]` error.

In [ ]:
# Per-clip dataset:
session = fo.launch_app(dataset)

# To view the low-noise per-species means (32 points) from cell 8c instead, run:
#     session = fo.launch_app(sds)
# then open the Embeddings panel, brain key viz_clap_embeddings, colour by family.
#
# In the grid, audio samples render as spectrograms (via @roboav8r/fiftyone-audio-toolkit).
# Double-click a tile -> modal with spectrogram + <audio> player + playhead. Add the
# "Audio Embeddings Search" panel from the + menu for query-by-sample/clip/text search.
#
# If you launched from a script (not Jupyter), add session.wait() to keep the App alive.
session


### Cleanup (optional)

```python
fo.delete_dataset(DATASET_NAME)
fo.delete_dataset(f"{DATASET_NAME}-species")   # from cell 8c
import shutil; shutil.rmtree(PLACEHOLDER_DIR, ignore_errors=True)
```

To remove the plugin too:

```bash
fiftyone plugins delete @roboav8r/fiftyone-audio-toolkit
```

Or just delete the whole `.venv-fiftyone-demo` folder and the cloned repo — the demo leaves
nothing behind except the plugin (in `~/fiftyone/__plugins__/`) and any placeholder PNGs.